# Set up for dataset and model

Package installation, loading, and dataloaders. There's also a resnet18 model defined.

In [38]:
# !pip install tensorboardX

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
from tqdm import tqdm

from torchvision import datasets, transforms
# from tensorboardX import SummaryWriter

use_cuda = True
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
train_dataset = datasets.CIFAR10('cifar10_data/', train=True, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))
test_dataset = datasets.CIFAR10('cifar10_data/', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)



In [39]:

def tp_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return .5 * (x + delta) * (1 - ind1) * (1 - ind2) + x * ind2

def tp_smoothed_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return (x + delta) ** 2 / (4 * delta) * (1 - ind1) * (1 - ind2) + x * ind2

class Normalize(nn.Module):
    def __init__(self, mu, std):
        super(Normalize, self).__init__()
        self.mu, self.std = mu, std

    def forward(self, x):
        return (x - self.mu) / self.std

class IdentityLayer(nn.Module):
    def forward(self, inputs):
        return inputs
    
class PreActBlock(nn.Module):
    '''Pre-activation version of the BasicBlock.'''
    expansion = 1

    def __init__(self, in_planes, planes, bn, learnable_bn, stride=1, activation='relu'):
        super(PreActBlock, self).__init__()
        self.collect_preact = True
        self.activation = activation
        self.avg_preacts = []
        self.bn1 = nn.BatchNorm2d(in_planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=not learnable_bn)
        self.bn2 = nn.BatchNorm2d(planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=not learnable_bn)

        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=not learnable_bn)
            )

    def act_function(self, preact):
        if self.activation == 'relu':
            act = F.relu(preact)
        elif self.activation[:6] == '3prelu':
            act = tp_relu(preact, delta=float(self.activation.split('relu')[1]))
        elif self.activation[:8] == '3psmooth':
            act = tp_smoothed_relu(preact, delta=float(self.activation.split('smooth')[1]))
        else:
            assert self.activation[:8] == 'softplus'
            beta = int(self.activation.split('softplus')[1])
            act = F.softplus(preact, beta=beta)
        return act

    def forward(self, x):
        out = self.act_function(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x  # Important: using out instead of x
        out = self.conv1(out)
        out = self.conv2(self.act_function(self.bn2(out)))
        out += shortcut
        return out

class PreActResNet(nn.Module):
    def __init__(self, block, num_blocks, n_cls, cuda=True, half_prec=False,
        activation='relu', fts_before_bn=False, normal='none'):
        super(PreActResNet, self).__init__()
        self.bn = True
        self.learnable_bn = True  # doesn't matter if self.bn=False
        self.in_planes = 64
        self.avg_preact = None
        self.activation = activation
        self.fts_before_bn = fts_before_bn
        if normal == 'cifar10':
            self.mu = torch.tensor((0.4914, 0.4822, 0.4465)).view(1, 3, 1, 1)
            self.std = torch.tensor((0.2471, 0.2435, 0.2616)).view(1, 3, 1, 1)
        else:
            self.mu = torch.tensor((0.0, 0.0, 0.0)).view(1, 3, 1, 1)
            self.std = torch.tensor((1.0, 1.0, 1.0)).view(1, 3, 1, 1)
            print('no input normalization')
        if cuda:
            self.mu = self.mu.cuda()
            self.std = self.std.cuda()
        if half_prec:
            self.mu = self.mu.half()
            self.std = self.std.half()

        self.normalize = Normalize(self.mu, self.std)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=not self.learnable_bn)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.bn = nn.BatchNorm2d(512 * block.expansion)
        self.linear = nn.Linear(512*block.expansion, n_cls)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, self.bn, self.learnable_bn, stride, self.activation))
            # layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x, return_features=False):
        for layer in [*self.layer1, *self.layer2, *self.layer3, *self.layer4]:
            layer.avg_preacts = []

        out = self.normalize(x)
        out = self.conv1(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        if return_features and self.fts_before_bn:
            return out.view(out.size(0), -1)
        out = F.relu(self.bn(out))
        if return_features:
            return out.view(out.size(0), -1)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)

        return out


def PreActResNet18(n_cls, cuda=True, half_prec=False, activation='relu', fts_before_bn=False,
    normal='none'):
    #print('initializing PA RN-18 with act {}, normal {}'.format())
    return PreActResNet(PreActBlock, [2, 2, 2, 2], n_cls=n_cls, cuda=cuda, half_prec=half_prec,
        activation=activation, fts_before_bn=fts_before_bn, normal=normal)


# intialize the model
model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
model.eval()
torch.save(model.state_dict(), 'original_untrained_model.pth')

no input normalization


# Implement the Attacks

Functions are given a simple useful signature that you can start with. Feel free to extend the signature as you see fit.

You may find it useful to create a 'batched' version of PGD that you can use to create the adversarial attack.

# Evaluate Single and Multi-Norm Robust Accuracy

In this section, we evaluate the model on the Linf and L2 attacks as well as union accuracy.

In [40]:
def pgd_linf_untargeted(model, x, labels, k, eps, eps_step):
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        loss = ce_loss(output, labels) 
        loss.backward()
        
        # Gradient ascent step
        adv_x = adv_x.detach() + eps_step * adv_x.grad.data.sign()
        
        # Project back to epsilon ball
        delta = adv_x - x
        delta = torch.clamp(delta, min=-eps, max=eps)
        adv_x = torch.clamp(x + delta, min=0, max=1).detach()
   
    return adv_x

In [41]:
def pgd_l2_untargeted(model, x, labels, k, eps, eps_step):
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    adv_x.requires_grad_(True) 
    for _ in range(k):
          adv_x.requires_grad_(True)
          model.zero_grad()
          output = model(adv_x)
          batch_size = x.size()[0]
          # TODO: Calculate the loss
          loss = ce_loss(output, labels)
          loss.backward()
          # TODO: compute the adv_x
          # find delta, clamp with eps, project delta to the l2 ball
          # HINT: https://github.com/Harry24k/adversarial-attacks-pytorch/blob/master/torchattacks/attacks/pgdl2.py 
          grad = adv_x.grad.data
          grad_norm = torch.norm(grad.view(batch_size, -1), p=2, dim=1)
          normalized_grad = grad / grad_norm.view(batch_size, 1, 1, 1)
          
          adv_x = adv_x.detach() + eps_step * normalized_grad
          
          delta = adv_x - x
          delta_norm = torch.norm(delta.view(batch_size, -1), p=2, dim=1)
          factor = torch.min(eps / delta_norm, torch.ones_like(delta_norm))
          delta = delta * factor.view(-1, 1, 1, 1)
          adv_x = torch.clamp(x + delta, min=0, max=1).detach()
   
    return adv_x

# Evaluate Single and Multi-Norm Robust Accuracy

In this section, we evaluate the model on the Linf and L2 attacks as well as union accuracy.

In [42]:
def test_model_on_single_attack(model, attack='pgd_linf', eps=0.1):
    model.eval()
    tot_test, tot_acc = 0.0, 0.0
    tot_test_o, tot_acc_o = 0.0, 0.0
    k = 10
    for batch_idx, (x_batch, y_batch) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        if attack == 'pgd_linf':
            # TODO: get x_adv untargeted pgd linf with eps, and eps_step=eps/4
            x_adv = pgd_linf_untargeted(model, x_batch, y_batch, k, eps, eps_step=eps/4)
        elif attack == 'pgd_l2':
            # TODO: get x_adv untargeted pgd l2 with eps, and eps_step=eps/4
            x_adv = pgd_l2_untargeted(model, x_batch, y_batch, k, eps, eps_step=eps/4)
        else:
            pass
        
        # get the testing accuracy and update tot_test and tot_acc
        with torch.no_grad():
            output = model(x_adv)
            pred = torch.max(output, dim=1)[1]
            tot_acc += (pred == y_batch).sum().item()
            tot_test += y_batch.size(0)
            
            output_o = model(x_batch)
            pred_o = torch.max(output_o, dim=1)[1]
            tot_acc_o += (pred_o == y_batch).sum().item()
            tot_test_o += y_batch.size(0)  
            
    print('Robust accuracy %.5lf' % (tot_acc/tot_test), f'on {attack} attack with eps = {eps}')
    print('Standard accuracy %.5lf' % (tot_acc_o/tot_test_o), f'on original attack with eps = {eps}')

## Single-Norm Robust Accuracy

In [43]:
'''
# Evaluate on Linf attack with different models with eps = 8/255
model.load_state_dict(torch.load('models/pretr_Linf.pth'))
# Evaluate on Linf attack with model 1 with eps = 8/255
test_model_on_single_attack(model, attack='pgd_linf', eps=8/255) 

model.load_state_dict(torch.load('models/pretr_L2.pth'))
# Evaluate on Linf attack with model 2 with eps = 8/255
test_model_on_single_attack(model, attack='pgd_linf', eps=8/255) 

model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
# Evaluate on Linf attack with model 3 with eps = 8/255
test_model_on_single_attack(model, attack='pgd_linf', eps=8/255) 
'''

"\n# Evaluate on Linf attack with different models with eps = 8/255\nmodel.load_state_dict(torch.load('models/pretr_Linf.pth'))\n# Evaluate on Linf attack with model 1 with eps = 8/255\ntest_model_on_single_attack(model, attack='pgd_linf', eps=8/255) \n\nmodel.load_state_dict(torch.load('models/pretr_L2.pth'))\n# Evaluate on Linf attack with model 2 with eps = 8/255\ntest_model_on_single_attack(model, attack='pgd_linf', eps=8/255) \n\nmodel.load_state_dict(torch.load('models/pretr_RAMP.pth'))\n# Evaluate on Linf attack with model 3 with eps = 8/255\ntest_model_on_single_attack(model, attack='pgd_linf', eps=8/255) \n"

In [44]:
'''
# Evaluate on L2 attack with different models with eps = 0.75
model.load_state_dict(torch.load('models/pretr_Linf.pth'))
# Evaluate on Linf attack with model 1 with eps = 0.75
test_model_on_single_attack(model, attack='pgd_l2', eps=0.75) 

model.load_state_dict(torch.load('models/pretr_L2.pth'))
# Evaluate on Linf attack with model 2 with eps = 0.75
test_model_on_single_attack(model, attack='pgd_l2', eps=0.75) 

model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
# Evaluate on Linf attack with model 3 with eps = 0.75
test_model_on_single_attack(model, attack='pgd_l2', eps=0.75) 
'''

"\n# Evaluate on L2 attack with different models with eps = 0.75\nmodel.load_state_dict(torch.load('models/pretr_Linf.pth'))\n# Evaluate on Linf attack with model 1 with eps = 0.75\ntest_model_on_single_attack(model, attack='pgd_l2', eps=0.75) \n\nmodel.load_state_dict(torch.load('models/pretr_L2.pth'))\n# Evaluate on Linf attack with model 2 with eps = 0.75\ntest_model_on_single_attack(model, attack='pgd_l2', eps=0.75) \n\nmodel.load_state_dict(torch.load('models/pretr_RAMP.pth'))\n# Evaluate on Linf attack with model 3 with eps = 0.75\ntest_model_on_single_attack(model, attack='pgd_l2', eps=0.75) \n"

## Multi-Norm Robust Accuracy

In [45]:
def test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75):
    model.eval()
    tot_test, tot_acc = 0.0, 0.0
    tot_test_o, tot_acc_o = 0.0, 0.0
    k = 10
    for batch_idx, (x_batch, y_batch) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        # TODO: get x_adv_linf and x_adv_l2 untargeted pgd linf and l2 with eps, and eps_step=eps/4
        x_adv_linf = pgd_linf_untargeted(model, x_batch, y_batch, k, eps_linf, eps_step=eps_linf/4)
        x_adv_l2 = pgd_l2_untargeted(model, x_batch, y_batch, k, eps_l2, eps_step = eps_l2/4)
        
        ## calculate union accuracy: correct only if both attacks are correct
        
        out = model(x_adv_linf)
        pred_linf = torch.max(out, dim=1)[1]
        out = model(x_adv_l2)
        pred_l2 = torch.max(out, dim=1)[1]
        
        # TODO: get the testing accuracy with multi-norm robustness and update tot_test and tot_acc
        tot_acc += ((pred_linf == y_batch) & (pred_l2 == y_batch)).sum().item()
        tot_test += y_batch.size(0)
        
        output_o = model(x_batch)
        pred_o = torch.max(output_o, dim=1)[1]
        tot_acc_o += (pred_o == y_batch).sum().item()
        tot_test_o += y_batch.size(0)  
            
    print('Robust accuracy %.5lf' % (tot_acc/tot_test), f'on multi attacks')
    print('Standard accuracy %.5lf' % (tot_acc_o/tot_test_o), f'on original attack')

In [46]:
''' 
# Evaluate on multi-norm attacks with different models with eps_linf = 8./255, eps_l2 = 0.75
model.load_state_dict(torch.load('models/pretr_Linf.pth'))
# Evaluate on multi attacks with model 1
test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75)

model.load_state_dict(torch.load('models/pretr_L2.pth'))
# Evaluate on multi attacks with model 2
test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75)

model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
# Evaluate on multi attacks with model 3
test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75)
'''


" \n# Evaluate on multi-norm attacks with different models with eps_linf = 8./255, eps_l2 = 0.75\nmodel.load_state_dict(torch.load('models/pretr_Linf.pth'))\n# Evaluate on multi attacks with model 1\ntest_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75)\n\nmodel.load_state_dict(torch.load('models/pretr_L2.pth'))\n# Evaluate on multi attacks with model 2\ntest_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75)\n\nmodel.load_state_dict(torch.load('models/pretr_RAMP.pth'))\n# Evaluate on multi attacks with model 3\ntest_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75)\n"

Standard Accuracy Evaluation Function

In [47]:
def evaluate_standard_accuracy(model, test_loader, device):
    """Evaluate standard accuracy on clean test data"""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return 100. * correct / total

In [48]:
def evaluate_robust_accuracy(model, test_loader, device, eps, k=10, eps_step=None):
    """Evaluate robust accuracy against PGD attack"""
    # CRITICAL: For FGSM (k=1), eps_step MUST equal eps (one full step)
    # For PGD (k>1), use smaller steps (eps/4)
    if eps_step is None:
        if k == 1:
            eps_step = eps  # FGSM: ONE full epsilon step
        else:
            eps_step = eps / 4  # PGD: multiple smaller steps
    
    model.eval()
    correct = 0
    total = 0
    
    print(f"  Attack config: k={k}, eps={eps:.4f}, eps_step={eps_step:.4f}")
    
    for inputs, targets in tqdm(test_loader, desc=f'Attacking (k={k})'):
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Generate adversarial examples
        adv_inputs = pgd_linf_untargeted(model, inputs, targets, k, eps, eps_step)
        
        # Evaluate on adversarial examples
        with torch.no_grad():
            outputs = model(adv_inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return 100. * correct / total

In [49]:
def adversarial_training(model, train_loader, test_loader, device, 
                        epochs=10, lr=0.1, eps=8/255, k=7, eps_step=None):
    """
    PGD-based Adversarial Training
    
    Key: Train on adversarial examples (adv_x) with correct labels (y_batch)
    
    Args:
        model: Neural network model
        train_loader: Training data loader
        test_loader: Test data loader
        device: cuda or cpu
        epochs: Number of training epochs
        lr: Learning rate
        eps: Epsilon for PGD attack (perturbation budget)
        k: Number of PGD steps during training
        eps_step: Step size for each PGD iteration
    """
    if eps_step is None:
        eps_step = eps / 4
    
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, 
                                               milestones=[int(epochs*0.5), int(epochs*0.75)], 
                                               gamma=0.1)
    criterion = nn.CrossEntropyLoss()
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'standard_acc': [],
        'robust_acc': []
    }
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for batch_idx, (inputs, targets) in enumerate(pbar):
            inputs, targets = inputs.to(device), targets.to(device)
            model.eval()
            adv_inputs = pgd_linf_untargeted(model, inputs, targets, k, eps, eps_step)
            
            model.train()
            
            optimizer.zero_grad()
            outputs = model(adv_inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            # Track metrics
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
            pbar.set_postfix({
                'Loss': f'{train_loss/(batch_idx+1):.3f}',
                'Acc': f'{100.*correct/total:.2f}%'
            })
        
        scheduler.step()
        
        # Record training metrics
        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(100. * correct / total)
        
        # Evaluate after each epoch
        print(f'\nEpoch {epoch+1} Results:')
        standard_acc = evaluate_standard_accuracy(model, test_loader, device)
        robust_acc = evaluate_robust_accuracy(model, test_loader, device, eps, k=10, eps_step=eps/4)
        
        history['standard_acc'].append(standard_acc)
        history['robust_acc'].append(robust_acc)
        
        print(f'Standard Accuracy: {standard_acc:.2f}%')
        print(f'Robust Accuracy (eps={eps:.4f}): {robust_acc:.2f}%')
        print('-' * 60)
    
    return model, history

In [50]:
# Part a
epsilon_values = [0.01, 0.05, 0.1]
all_results = {}

print(f"Testing {len(epsilon_values)} epsilon values: {epsilon_values}")
print(f"Using k={10} PGD steps during training\n")

for eps in epsilon_values:
    print(f"\n{'='*80}")
    print(f"Training with epsilon = {eps:.4f}")
    print(f"{'='*80}\n")
    
    # Initialize new model for each epsilon
    model_adv = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
    
    # Train model with adversarial training
    trained_model, history = adversarial_training(
        model=model_adv,
        train_loader=train_loader,
        test_loader=test_loader,
        device=device,
        epochs=10,  # Increase to 50-100 for better results
        lr=0.1,
        eps=eps,
        k=10,  # Using k=10 PGD steps
        eps_step=eps/4
    )
    
    # Save the trained model with epsilon value in filename
    model_name = f'adv_trained_eps_{eps:.4f}.pth'
    torch.save(trained_model.state_dict(), model_name)
    print(f"\nModel saved as: {model_name}")
    
    # Store results with epsilon as key
    all_results[f'eps_{eps:.4f}'] = history
    
    # Final evaluation
    print(f"\n{'='*80}")
    print(f"Final Results for eps={eps:.4f}:")
    print(f"Standard Accuracy: {history['standard_acc'][-1]:.2f}%")
    print(f"Robust Accuracy: {history['robust_acc'][-1]:.2f}%")
    print(f"Accuracy Drop: {history['standard_acc'][-1] - history['robust_acc'][-1]:.2f}%")
    print(f"{'='*80}\n")

print("\n" + "="*80)
print("TRAINING COMPLETE FOR ALL EPSILON VALUES")
print("="*80)

Testing 3 epsilon values: [0.01, 0.05, 0.1]
Using k=10 PGD steps during training


Training with epsilon = 0.0100

no input normalization


Epoch 1/10: 100%|██████████| 782/782 [02:35<00:00,  5.04it/s, Loss=1.881, Acc=29.27%]



Epoch 1 Results:
  Attack config: k=10, eps=0.0100, eps_step=0.0025


Attacking (k=10): 100%|██████████| 157/157 [00:27<00:00,  5.81it/s]


Standard Accuracy: 37.44%
Robust Accuracy (eps=0.0100): 29.78%
------------------------------------------------------------


...

Standard Accuracy: 23.52%
Robust Accuracy (eps=0.1000): 16.03%
------------------------------------------------------------


Epoch 10/10: 100%|██████████| 782/782 [01:18<00:00,  9.97it/s, Loss=2.226, Acc=16.30%]



Epoch 10 Results:
  Attack config: k=10, eps=0.1000, eps_step=0.0250


Attacking (k=10): 100%|██████████| 157/157 [00:14<00:00, 10.82it/s]


Standard Accuracy: 23.51%
Robust Accuracy (eps=0.1000): 16.23%
------------------------------------------------------------

Model saved as: adv_trained_eps_0.1000.pth

Final Results for eps=0.1000:
Standard Accuracy: 23.51%
Robust Accuracy: 16.23%
Accuracy Drop: 7.28%


TRAINING COMPLETE FOR ALL EPSILON VALUES


In [51]:
# Train a standard (non-adversarial) model for comparison in part b
print("Training standard model for comparison...")
standard_model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
optimizer = optim.SGD(standard_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

# Standard training
num_epochs = 10
for epoch in range(num_epochs):
    standard_model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f'Standard Training {epoch+1}/{num_epochs}')
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = standard_model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        pbar.set_postfix({
            'Loss': f'{train_loss/(len(pbar)):.3f}',
            'Acc': f'{100.*correct/total:.2f}%'
        })
    
    # Evaluate
    std_acc = evaluate_standard_accuracy(standard_model, test_loader, device)

torch.save(standard_model.state_dict(), 'standard_trained.pth')

Training standard model for comparison...
no input normalization


Standard Training 3/10:  33%|███▎      | 258/782 [00:05<00:09, 53.08it/s, Loss=0.385, Acc=59.23%]

Standard Training 10/10: 100%|██████████| 782/782 [00:16<00:00, 46.86it/s, Loss=0.769, Acc=73.28%]


In [52]:
# Part b

import os
import torch

standard_model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
standard_model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
standard_model.eval()

test_eps = 0.05  # The epsilon for adversarial training

adv_trained_model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
adv_trained_model.load_state_dict(torch.load(f'adv_trained_eps_{test_eps:.4f}.pth'))
adv_trained_model.eval()

standard_clean = evaluate_standard_accuracy(standard_model, test_loader, device)

adv_clean = evaluate_standard_accuracy(adv_trained_model, test_loader, device)

no input normalization
no input normalization


In [53]:
# Complete Part b

import torch
import matplotlib.pyplot as plt
import numpy as np

standard_model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
standard_model.load_state_dict(torch.load('standard_trained.pth'))
standard_model.eval()

test_eps = 0.05
adv_trained_model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
adv_trained_model.load_state_dict(torch.load(f'adv_trained_eps_{test_eps:.4f}.pth'))
adv_trained_model.eval()

epsilon_values = [0.01, 0.05, 0.1]

results = {
    'epsilons': epsilon_values,
    'standard': {'clean': 0, 'robust': []},
    'adv_trained': {'clean': 0, 'robust': []}
}

results['standard']['clean'] = evaluate_standard_accuracy(standard_model, test_loader, device)
results['adv_trained']['clean'] = evaluate_standard_accuracy(adv_trained_model, test_loader, device)

clean_diff = results['standard']['clean'] - results['adv_trained']['clean']
for i, eps in enumerate(epsilon_values, 1):
    
    # Attack clean model
    std_robust = evaluate_robust_accuracy(
        standard_model, test_loader, device,
        eps=eps, k=1  # FGSM: k=1, eps_step will be set to eps automatically
    )
    results['standard']['robust'].append(std_robust)
    
    # Attack adv model
    adv_robust = evaluate_robust_accuracy(
        adv_trained_model, test_loader, device,
        eps=eps, k=1  # FGSM: k=1
    )
    results['adv_trained']['robust'].append(adv_robust)
    

    std_drop = results['standard']['clean'] - std_robust
    adv_drop = results['adv_trained']['clean'] - adv_robust
    improvement = adv_robust - std_robust



no input normalization
no input normalization
  Attack config: k=1, eps=0.0100, eps_step=0.0100


Attacking (k=1): 100%|██████████| 157/157 [00:03<00:00, 46.69it/s]


  Attack config: k=1, eps=0.0100, eps_step=0.0100


Attacking (k=1): 100%|██████████| 157/157 [00:03<00:00, 51.06it/s]


  Attack config: k=1, eps=0.0500, eps_step=0.0500


...

Attacking (k=1): 100%|██████████| 157/157 [00:03<00:00, 45.34it/s]


  Attack config: k=1, eps=0.1000, eps_step=0.1000


Attacking (k=1): 100%|██████████| 157/157 [00:03<00:00, 45.52it/s]


  Attack config: k=1, eps=0.1000, eps_step=0.1000


Attacking (k=1): 100%|██████████| 157/157 [00:03<00:00, 45.88it/s]


In [54]:
print(f"\nStandardStandard Accuracy (No Attack):")
print(f"  Standard Model: {results['standard']['clean']:.2f}%")
print(f"  Adv Trained Model: {results['adv_trained']['clean']:.2f}%")
print(f"  Accuracy Trade-off: {results['standard']['clean'] - results['adv_trained']['clean']:.2f}%")

print(f"\n" + "="*80)
print("FGSM Attack Effectiveness")
print("="*80)

for i, eps in enumerate(results['epsilons']):
    std_acc = results['standard']['robust'][i]
    adv_acc = results['adv_trained']['robust'][i]
    std_drop = results['standard']['clean'] - std_acc
    adv_drop = results['adv_trained']['clean'] - adv_acc
    improvement = adv_acc - std_acc
    
    print(f"\nEpsilon = {eps:.4f}")
    print(f"  Standard Model: {std_acc:.2f}% (drop: {std_drop:.2f}%)")
    print(f"  Adv Trained Model: {adv_acc:.2f}% (drop: {adv_drop:.2f}%)")
    print(f"  Improvement: +{improvement:.2f}%")
    
    if adv_drop > 0.1:
        effectiveness = std_drop / adv_drop
        print(f"  Attack is {effectiveness:.2f}x MORE effective on standard model")
    else:
        print(f"  Adv trained model is nearly perfectly robust at this epsilon")

print("\n" + "="*80)
print("KEY OBSERVATIONS:")
print("="*80)
print("""

""".format(results['standard']['clean'] - results['adv_trained']['clean']))


StandardStandard Accuracy (No Attack):
  Standard Model: 26.07%
  Adv Trained Model: 40.17%
  Accuracy Trade-off: -14.10%

FGSM Attack Effectiveness

Epsilon = 0.0100
  Standard Model: 5.72% (drop: 20.35%)
  Adv Trained Model: 37.15% (drop: 3.02%)
  Improvement: +31.43%
  Attack is 6.74x MORE effective on standard model

Epsilon = 0.0500
  Standard Model: 2.11% (drop: 23.96%)
  Adv Trained Model: 25.50% (drop: 14.67%)
  Improvement: +23.39%
  Attack is 1.63x MORE effective on standard model

Epsilon = 0.1000
  Standard Model: 3.16% (drop: 22.91%)
  Adv Trained Model: 14.60% (drop: 25.57%)
  Improvement: +11.44%
  Attack is 0.90x MORE effective on standard model

KEY OBSERVATIONS:



